# Fairness as a verifiable property (stretch)

*Notebook 2 of 2 · about 20 minutes · run notebook 1 first, or at least its setup cell.*

A loan-approval network takes six numbers about an applicant — income, debt-to-income ratio, years of credit history, past defaults, age, and a **protected attribute** (0/1) — and returns approve/deny scores. We would like to *prove*:

> for every pair of applicants that differ **only** in the protected attribute, the decision is the same.

That is a statement about two evaluations of the network at once. Solvers such as Marabou evaluate a network once per query, so the property cannot be written directly. The trick: build a **twin network** — two weight-shared copies side by side as *one* ONNX graph, taking a pair (A ++ B) as a single 12-number input and returning (scores A ++ scores B). Now the fairness property is an ordinary property of one network. (Idea: Athavale et al., *Verifying Global Two-Safety Properties in Neural Networks with Confidence*, CAV 2024.)

Everything is synthetic. The historical approval labels penalise the protected group on purpose — that is the point.

## 0 · Setup (skip if you ran notebook 1 in this session)

**Expected output:** as in notebook 1, ending with `OK - ready.`

In [1]:
# --- Setup (idempotent; safe to re-run) -------------------------------------------------------------
# Colab's kernel is Python 3.12 and the Marabou solver only ships wheels for Python 3.8-3.11 (x86_64),
# so the toolchain lives in a separate Python 3.11 environment created by environment/colab_bootstrap.sh.
# We never import the verifier into this kernel; we call the `vehicle` command-line tool.
import os, sys, subprocess, pathlib, shutil, time

REPO_URL = "https://github.com/Yiergot/acaira-2026-nn-verification"   # <- repository (placeholder until published)
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

here = pathlib.Path.cwd()
if (here / "specs" / "triage.vcl").exists():
    ROOT = here
elif (here.parent / "specs" / "triage.vcl").exists():
    ROOT = here.parent
else:
    if not pathlib.Path("acaira-2026-nn-verification").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    ROOT = pathlib.Path("acaira-2026-nn-verification").resolve()
os.chdir(ROOT)
print("working directory:", ROOT)

if shutil.which("vehicle") and shutil.which("Marabou"):
    print("toolchain already present:", shutil.which("vehicle"))
else:
    VENV = "/content/venv" if IN_COLAB else str(ROOT.parent / ".venv-nnv")
    t0 = time.time()
    subprocess.run(["bash", "environment/colab_bootstrap.sh"], env={**os.environ, "VENV": VENV}, check=True)
    os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
    print(f"bootstrap took {time.time() - t0:.0f} s")

# small helpers for THIS kernel (plain Python, no verifier)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "idx2numpy", "matplotlib", "pandas"],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(ROOT / "src"))
print("vehicle", subprocess.run(["vehicle", "--version"], capture_output=True, text=True).stdout.strip(),
      "| Marabou at", shutil.which("Marabou"))
print("OK - ready.")

working directory: /work
bootstrap took 1 s
vehicle 0.27.1 | Marabou at /content/venv/bin/Marabou
OK - ready.


In [2]:
# --- Helpers: run the verifier and read its answers ------------------------------------------------
import re, subprocess, time
import numpy as np, pandas as pd, onnxruntime as ort
import news2

ANSI = re.compile(r"\x1b\[[0-9;]*m")
NOISE = ("Warning", "strict inequalit", "Unfortunately", "In order to provide", "not sound", "excluded middle",
         "issues/74", "floating point", "unexpected behaviour", "queries")   # progress bars end with "queries"

def run_vehicle(args, wall=None):
    """Run `vehicle <args>`, return (cleaned output lines, seconds). `wall` = hard wall-clock cap in seconds:
    the solver's own --timeout is not honoured on some degenerate queries, so we never wait for ever."""
    t0 = time.time()
    try:
        p = subprocess.run(["vehicle", *args], capture_output=True, text=True, timeout=wall)
        raw = p.stdout + p.stderr
    except subprocess.TimeoutExpired as e:
        raw = ((e.stdout or b"").decode() if isinstance(e.stdout, bytes) else (e.stdout or "")) + \
              "\n    result: ? - wall-clock cap reached, solver killed\n"
    raw = ANSI.sub("", raw).replace("\r", "\n")
    lines = [l.rstrip() for l in raw.split("\n") if l.strip() and not any(n in l for n in NOISE)]
    return lines, time.time() - t0

def verify(spec, network, properties=(), datasets=None, parameters=None, network_name="triage", timeout=30, show=True):
    """Verify `properties` of `spec` for the ONNX file `network`. Returns a dict with statuses, counterexamples, summary."""
    args = ["verify", "-s", spec, "-n", f"{network_name}:{network}", "--solver", "Marabou", "-a", f"--timeout={timeout}"]
    for prop in properties: args += ["-y", prop]
    for k, v in (datasets or {}).items(): args += ["-d", f"{k}:{v}"]
    for k, v in (parameters or {}).items(): args += ["-p", f"{k}:{v}"]
    lines, secs = run_vehicle(args, wall=3 * timeout + 30)
    statuses, ces, summary, i = [], [], {}, 0
    while i < len(lines):
        l = lines[i]
        if "result:" in l:
            if "proved no counterexample" in l or "proved no witness" in l: statuses.append("verified"); ces.append(None)
            elif "found a counterexample" in l or "found a witness" in l:
                statuses.append("falsified")
                vec = None
                if i + 1 < len(lines) and re.match(r"\s*[a-zA-Z]+:\s*\[", lines[i + 1]):
                    txt = lines[i + 1]; j = i + 1
                    while "]" not in txt and j + 1 < len(lines): j += 1; txt += lines[j]
                    vec = [float(v) for v in re.search(r"\[(.*)\]", txt).group(1).split(",")]
                ces.append(vec)
            elif "timed out" in l or "wall-clock" in l: statuses.append("timeout"); ces.append(None)
            else: statuses.append("other"); ces.append(None)
        m = re.match(r"\s*(verified|falsified|timed-out|errored):\s*(\d+)/(\d+)", l)
        if m: summary[m.group(1)] = (int(m.group(2)), int(m.group(3)))
        i += 1
    names = list(properties) if properties and not summary else [f"{properties[0] if properties else 'item'}!{k}" for k in range(len(statuses))]
    if show:
        glyph = {"verified": "VERIFIED  ", "falsified": "FALSIFIED ", "timeout": "TIMEOUT   ", "other": "?         "}
        for n, s, c in zip(names, statuses, ces):
            print(f"{glyph[s]} {n}" + (f"   counterexample: {np.round(c, 2).tolist()}" if c else ""))
        if summary: print("summary:", {k: f"{a}/{b}" for k, (a, b) in summary.items()})
        print(f"({secs:.1f} s)")
    return {"statuses": statuses, "counterexamples": ces, "summary": summary, "seconds": secs, "lines": lines}

_sessions = {}
def predict(model_path, x):
    """Class scores of an ONNX network for one input vector (list of floats)."""
    if model_path not in _sessions:
        _sessions[model_path] = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    s = _sessions[model_path]; name = s.get_inputs()[0].name
    return s.run(None, {name: np.asarray(x, dtype=np.float32)[None]})[0][0]

def show_patient(x, models=("v1", "v2", "v3")):
    """Print a counterexample as a patient chart: values, units, NEWS2 points, and what each model advises."""
    x = np.asarray(x, dtype=float)
    pts = news2.component_scores(x[None])[0]
    rows = [(f, round(v, 2), news2.UNITS[f], int(p)) for f, v, p in zip(news2.FEATURES, x, pts)]
    print(pd.DataFrame(rows, columns=["parameter", "value", "unit", "NEWS2 points"]).to_string(index=False))
    agg = int(pts.sum()); rule = news2.CLASSES[int(news2.triage_class(x[None])[0])]
    print(f"\naggregate NEWS2 score {agg}; highest single parameter {int(pts.max())}  ->  the rule says: {rule.upper()}")
    for m in models:
        sc = predict(f"models/triage-{m}.onnx", x)
        top2 = np.sort(sc)[-2:]
        tie = "  <- TIE: the two highest scores are equal to within rounding; the decision boundary passes through this patient" if top2[1] - top2[0] < 1e-3 else ""
        print(f"triage-{m} advises: {news2.CLASSES[int(np.argmax(sc))].upper():7s}  (scores low/medium/high = {np.round(sc, 2).tolist()}){tie}")
print("helpers loaded")

helpers loaded


## 1 · The data and the two models

`src/loan.py` generates 30 000 applicants. Creditworthiness is a simple function of the numbers; the **historical** approval label subtracts a penalty for the protected group. Two networks:

- `loan-v1` — trained on the historical labels with the protected attribute as an input;
- `loan-v2` — same, but the input weights for the attribute are zeroed after training: it *cannot* read the flag.

**Expected output:** approval rates by group in the labels (≈ 49 % vs ≈ 18 %) and for each model. Note that v2 still shows a gap (≈ 39 % vs ≈ 28 %): the protected group has lower incomes in this synthetic society, and income is a **proxy**.

In [3]:
loans = pd.read_csv("data/loan-synthetic.csv")
FEATS = ["income", "debt_ratio", "history_years", "defaults", "age", "protected"]
print("historical labels — approval rate by protected group:", loans.groupby("protected")["approved_historical"].mean().round(3).to_dict())
print("merit-only labels  — approval rate by protected group:", loans.groupby("protected")["approved_merit_only"].mean().round(3).to_dict())
sample = loans.sample(5000, random_state=1)
for m in ("v1", "v2"):
    pred = np.array([int(np.argmax(predict(f"models/loan-{m}.onnx", x))) for x in sample[FEATS].to_numpy(np.float32)])  # 1 = approve
    rates = pd.Series(pred).groupby(sample["protected"].to_numpy()).mean().round(3).to_dict()
    acc = (pred == sample["approved_historical"].to_numpy()).mean()
    print(f"loan-{m}: approval rate by group {rates}; accuracy vs historical labels {acc:.1%}")

historical labels — approval rate by protected group: {0.0: 0.488, 1.0: 0.179}
merit-only labels  — approval rate by protected group: {0.0: 0.488, 1.0: 0.399}
loan-v1: approval rate by group {0.0: 0.498, 1.0: 0.158}; accuracy vs historical labels 93.2%
loan-v2: approval rate by group {0.0: 0.392, 1.0: 0.28}; accuracy vs historical labels 88.7%


## 2 · The twin specification

`specs/loan-twin.vcl` (printed below). Positions 0–5 are applicant A, 6–11 applicant B. `sameExceptProtected` ties every field of A to B and sets A's flag to 0, B's to 1. Two one-directional properties: `fairAtoB` (if A is *clearly* approved, B is approved) and `fairBtoA`.

Why "clearly", with a `margin`? Verifiers work with closed sets and relax `>` to `>=`, so a pair sitting exactly on the decision boundary would be reported as a spurious counterexample. Asking for a small margin on one side removes that artefact — and is itself a small specification decision.

In [4]:
print(open("specs/loan-twin.vcl").read())

--------------------------------------------------------------------------------
-- Fairness as a verifiable property (stretch exercise)
--
-- A loan-approval network f takes one applicant and returns two scores
-- (approve, deny). We would like to state:
--
--     for all applicants a, b that differ ONLY in the protected attribute,
--     f approves a  <=>  f approves b            ("counterfactual fairness")
--
-- That property applies the network twice, which query-based verifiers such
-- as Marabou cannot express. The trick: build a "twin" network that is two
-- weight-shared copies of f side by side (models/loan-twin-*.onnx). It takes
-- a pair (a ++ b) as ONE input of 12 numbers and returns (f a ++ f b) as ONE
-- output of 4 numbers. Now the fairness property is an ordinary property.
-- (Idea from Athavale et al., CAV 2024, "two-safety" properties.)
--------------------------------------------------------------------------------

type Pair   = Tensor Real [12]   -- applicant A (0.

## 3 · Ask the verifier (the biased model)

**Expected output.** For `loan-twin-v1`: `fairAtoB` **FALSIFIED** within a couple of seconds, with a pair — identical numbers, protected 0 approved, protected 1 denied — and `fairBtoA` VERIFIED (the model never favours the protected group).

In [5]:
LOAN_FEATS = ["income (k£/yr)", "debt ratio", "history (years)", "defaults", "age", "protected"]
r = verify("specs/loan-twin.vcl", "models/loan-twin-v1.onnx", ["fairAtoB", "fairBtoA"], parameters={"margin": 0.1},
           network_name="twin", timeout=60)
for prop, ce in zip(["fairAtoB", "fairBtoA"], r["counterexamples"]):
    if ce:
        A, B = ce[:6], ce[6:]
        tab = pd.DataFrame({"field": LOAN_FEATS, "applicant A": np.round(A, 3), "applicant B": np.round(B, 3)})
        print(f"\ncounterexample pair for {prop}:\n" + tab.to_string(index=False))
        for name, x in (("A", A), ("B", B)):
            sc = predict("models/loan-v1.onnx", x)
            print(f"  loan-v1 on {name}: {'APPROVE' if sc[1] > sc[0] else 'DENY'}  (deny/approve scores {np.round(sc, 2).tolist()})")

FALSIFIED  fairAtoB   counterexample: [10.0, 0.0, 0.17, 0.0, 18.0, 0.0, 10.0, 0.0, 0.17, 0.0, 18.0, 1.0]
VERIFIED   fairBtoA
(3.8 s)

counterexample pair for fairAtoB:
          field  applicant A  applicant B
 income (k£/yr)       10.000       10.000
     debt ratio        0.000        0.000
history (years)        0.167        0.167
       defaults        0.000        0.000
            age       18.000       18.000
      protected        0.000        1.000
  loan-v1 on A: APPROVE  (deny/approve scores [-0.2199999988079071, -0.11999999731779099])
  loan-v1 on B: DENY  (deny/approve scores [1.9199999570846558, -2.630000114440918])


## 3b · The attribute-blind model: a proof without a solver

`loan-v2` had its first-layer weights on the protected column set to exactly zero after training. That is a proof of counterfactual fairness in one line: **the attribute has no path to the output**, so flipping it cannot change anything. The cell below checks the weights, and checks the claim numerically on 2,000 random pairs.

We deliberately do **not** run Marabou on `loan-twin-v2`. Two identical sub-networks side by side make the query degenerate: in our tests Marabou looped without terminating (not even honouring its own `--timeout`), and a variant with near-zero instead of zero weights returned a "counterexample" that, re-evaluated, did not violate the property. Complete solvers have numerical corners; knowing when *not* to reach for one is part of the craft. (If you want to see it, the `verify` helper has a wall-clock cap, so a call on `loan-twin-v2` ends with `timeout` after a few minutes rather than hanging.)

**Expected output:** `max |weight| on the protected column: 0.0` for v2 (and a clearly non-zero value for v1), then `0 of 2000 random pairs change decision` for v2.

In [6]:
import onnx
from onnx import numpy_helper
for m in ("v1", "v2"):
    g = onnx.load(f"models/loan-{m}.onnx").graph
    W1 = [numpy_helper.to_array(t) for t in g.initializer if t.dims and len(t.dims) == 2][0]   # first Gemm weight (out, in)
    print(f"loan-{m}: first-layer weight matrix {W1.shape}; max |weight| on the protected column: {np.abs(W1[:, 5]).max():.6f}")

rng = np.random.default_rng(0)
LO = np.array([10, 0.0, 0, 0, 18, 0]); HI = np.array([200, 1.0, 40, 5, 90, 1])
A = rng.uniform(LO, HI, size=(2000, 6)); A[:, 3] = np.round(A[:, 3]); A[:, 5] = 0; B = A.copy(); B[:, 5] = 1
for m in ("v1", "v2"):
    flips = sum(int(np.argmax(predict(f"models/loan-{m}.onnx", a)) != np.argmax(predict(f"models/loan-{m}.onnx", b))) for a, b in zip(A, B))
    print(f"loan-{m}: {flips} of 2000 random pairs change decision when only the protected attribute is flipped")

loan-v1: first-layer weight matrix (8, 6); max |weight| on the protected column: 0.564367
loan-v2: first-layer weight matrix (8, 6); max |weight| on the protected column: 0.000000
loan-v1: 229 of 2000 random pairs change decision when only the protected attribute is flipped
loan-v2: 0 of 2000 random pairs change decision when only the protected attribute is flipped


## 4 · So is v2 fair?

It is *individually* counterfactually fair, with a proof (a structural one). Its approval rates by group are still ≈ 39 % vs ≈ 28 %. Both statements are true. Verification answered exactly the question we asked — and no other. Whether the right question is "same decision for the same numbers" or "similar outcomes for the two groups" is a policy choice; the second is not a property of the network alone, it is a property of the network *and* the society that produced the data.

Legal anchors for the debate: the EU AI Act lists credit scoring of natural persons as **high-risk** (Annex III), so Article 15's "appropriate robustness" applies; in the UK, the Equality Act 2010 covers *indirect* discrimination, which is exactly what a proxy produces.

## 5 · Ordinary properties of the single network

`specs/loan.vcl` states two other kinds of property on `loan-v1/v2` directly: a **region** property ("a plainly strong applicant is approved, whatever the flag") and **local robustness** ("a small change in reported income never flips the decision") around ten evaluation applicants (eight confident, two borderline).

**Expected output:** run it and read what the verifier says — the region property may well be *falsified* at a corner of the box (income 200 k£, age 90, …) where the model has extrapolated: report that honestly, it is a finding about the model, not a bug in the verifier. Income robustness: the confident applicants stay stable up to several thousand pounds; the two borderline ones flip at the smallest ε.

In [7]:
for m in ("v1", "v2"):
    print(f"=== loan-{m}: strong applicants approved (protected = 0 / = 1) ===")
    r = verify("specs/loan.vcl", f"models/loan-{m}.onnx", ["strongApplicantsApproved0", "strongApplicantsApproved1"], network_name="loan", timeout=60)
    for prop, ce in zip(["strongApplicantsApproved0", "strongApplicantsApproved1"], r["counterexamples"]):
        if ce: print(f"  {prop} counterexample:", dict(zip(LOAN_FEATS, np.round(ce, 2).tolist())))
    print()
applicants = pd.read_csv("data/loan-eval-applicants.csv")
print(applicants.to_string()); print()
for eps in (0.5, 2, 10):
    r = verify("specs/loan.vcl", "models/loan-v2.onnx", ["incomeRobust"], network_name="loan",
               datasets={"applicants": "data/loan-eval-applicants.idx", "decisions": "data/loan-eval-decisions.idx"},
               parameters={"epsIncome": eps}, show=False)
    print(f"income +/- {eps} k£: {r['summary'].get('verified', (0, 10))[0]}/10 applicants provably stable   ({r['seconds']:.1f} s)")

=== loan-v1: strong applicants approved (protected = 0 / = 1) ===
VERIFIED   strongApplicantsApproved0
VERIFIED   strongApplicantsApproved1
(1.4 s)

=== loan-v2: strong applicants approved (protected = 0 / = 1) ===
VERIFIED   strongApplicantsApproved0
VERIFIED   strongApplicantsApproved1
(1.4 s)

   income  debt_ratio  history_years  defaults    age  protected model_v2_decision  margin_logits  borderline
0  200.00        0.07          17.85       1.0  33.08        0.0           approve         13.490       False
1  172.37        0.17          16.19       0.0  37.93        0.0           approve         12.622       False
2  151.94        0.18           8.35       0.0  29.29        0.0           approve         10.757       False
3  171.01        0.40           9.78       0.0  18.93        0.0           approve         10.682       False
4   19.81        0.34          10.87       4.0  40.78        1.0              deny        -14.953       False
5   15.95        0.78           8.24      

## 6 · What you can now say in a design review

- "Counterfactual fairness with respect to attribute *p* is a *property*; here is the proof / here is the pair that violates it."
- "Removing the attribute made the property trivially true and changed group outcomes by this much; the rest is proxies."
- "The verifier reasons about one network application; hyperproperties need an encoding (self-composition), and that encoding is part of the specification you should review."

Back to the worksheet: which of your project's fairness or safety requirements is a *property*, and which is a *policy*?